In [1]:
import json
import os
import torch
from datetime import datetime
from torch_geometric.loader import DataLoader

from extractor import PDBBindOrchestrator
from tokenizer import UniversalPDBBindDataset
from evaluator import Evaluator
from splitter import PDBBindSplitter
from utils import Utils
from model.model_builder import UHSMBuilder
from parsers.cnn_parser import CNNParser
from parsers.gnn_parser import GNNParser
from logger import log_info
from model.trainer import HybridTrainer

In [2]:
with open('config.json', 'r') as f:
    config = json.load(f)

In [3]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
exp_name = f"{config['experiment_name']}_{timestamp}"
log_info(f"Experiment name: {exp_name}", stage="EXPERIMENT")
exp_run_dir = f"runs/{exp_name}"
exp_data_dir = f"datasets/{exp_name}" # Индивидуальная папка для датасетов!

os.makedirs(exp_run_dir, exist_ok=True)
os.makedirs(exp_data_dir, exist_ok=True)
os.makedirs("data/base_datasets", exist_ok=True) # Глобальный кэш

[INFO][EXPERIMENT] Experiment name: CCC_TEST_20260424_022321


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log_info(f"Launch on: {device}", stage="EXPERIMENT")

[INFO][EXPERIMENT] Launch on: cpu


In [5]:
trio_str = config['model']['graph_encoder']['available']['trio']['protein_ligand_pocket_encoders']
parsers = []
for i, char in enumerate(trio_str):
    is_lig = (i == 1)
    if char == 'C': parsers.append(CNNParser(is_ligand=is_lig))
    elif char == 'G': parsers.append(GNNParser(is_ligand=is_lig))
    elif char == 'N': parsers.append(None)
    else: log_info("Unknown symbol {char} in the architecture description.", stage="EXPERIMENT")

[INFO][CNNParser] Initialized is_ligand=False
[INFO][CNNParser] Initialized is_ligand=True
[INFO][CNNParser] Initialized is_ligand=False


In [6]:
orchestrator = PDBBindOrchestrator(parsers, config)
# orchestrator.extract_subset("refined")
df_refined = orchestrator.build_dataset(subset="refined", fmt="pickle", save_dir=exp_data_dir)
df_core = orchestrator.build_dataset(subset="core", fmt="pickle", save_dir=exp_data_dir)

[INFO][BUILD] Starting parallel parsing on 8 cores...
100%|██████████| 4057/4057 [00:42<00:00, 96.41it/s] 
[INFO][BUILD] Success: 4056, Errors: 1
[WARNING][BUILD] Errors: Counter({"ligand_parse_error: Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8": 1})
[INFO][SAVE] Metadata saved to datasets/CCC_TEST_20260424_022321/pdbbind_refined_trio_meta.json
[INFO][SAVE] Dataset saved to datasets/CCC_TEST_20260424_022321/pdbbind_refined_trio.pickle (compression: None)
[INFO][BUILD] Starting parallel parsing on 8 cores...
100%|██████████| 290/290 [00:02<00:00, 103.96it/s]
[INFO][BUILD] Success: 289, Errors: 1
[WARNING][BUILD] Errors: Counter({"ligand_parse_error: Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8": 1})
[INFO][SAVE] Metadata saved to datasets/CCC_TEST_20260424_022321/pdbbind_core_trio_meta.json
[INFO][SAVE] Dataset saved to datasets/CCC_TEST_20260424_022321/pdbbind_core_trio.pickle (compression: None)


In [7]:
clean_refined = df_refined[~df_refined['pdb_id'].isin(df_core['pdb_id'])]

train_df, val_df = PDBBindSplitter.split(clean_refined, config["splitter"])

test_df = df_core

train_path = f"{exp_data_dir}/train.pickle"
val_path   = f"{exp_data_dir}/val.pickle"
test_path  = f"{exp_data_dir}/test_core.pickle"

train_df.to_pickle(train_path)
test_df.to_pickle(test_path)
val_df.to_pickle(val_path)

config["dataset"].update({
    "train_path": train_path,
    "val_path": val_path,
    "test_path": test_path,
})

[INFO][SPLIT] Strategy: random
[INFO][SPLIT] Total: 3767 | Train: 3202 | Val: 565


In [8]:
train_ds = UniversalPDBBindDataset(config["dataset"]["train_path"], config)
test_ds = UniversalPDBBindDataset(config["dataset"]["test_path"], config)
val_ds   = UniversalPDBBindDataset(config["dataset"]["val_path"], config)

train_loader = DataLoader(train_ds, batch_size=config['dataset']['batch_size'], shuffle=True)
test_loader = DataLoader(test_ds, batch_size=config['dataset']['batch_size'], shuffle=False)
val_loader   = DataLoader(val_ds, batch_size=config['dataset']['batch_size'], shuffle=False)

In [9]:
exp_dir = Utils.handle_metadata(config, train_ds, val_ds, test_ds)
model = UHSMBuilder.build_model_from_config(config)
evaluator = Evaluator(model, device)
trainer = HybridTrainer(model, evaluator, config, device)
best_epoch, best_val_r = trainer.train(train_loader, val_loader, exp_dir)
trainer.test(test_loader, exp_dir, best_epoch)

[INFO][UTILS] Файлы сохраняются сюда: runs/CCC_TEST_20260424_022422


TypeError: QuantumReUploadingLayer.__init__() got an unexpected keyword argument 'out_dim'